# Multimodal SFT: Image Classification

Fine-tune GPT-4o for Stanford Dogs classification. This notebook integrates the former separate latency notebook and covers deterministic splits, managed safety attrition, guarded training/reuse, per-class and robustness analysis, full latency distributions, and cost. `FOUNDRY_RUN_PAID_JOBS=false` and `FOUNDRY_RUN_LIVE_EVALUATION=false` by default.

## Setup

Copy `../.env.template` to `.env`, run `az login`, and install `../../../requirements.lock`. Raw downloads and generated outputs remain ignored.

In [ ]:
from __future__ import annotations
import hashlib, io, json, os, random, sys, time, urllib.request, zipfile
from collections import Counter
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
DEMO_ROOT=Path.cwd().resolve()
if DEMO_ROOT.name=="notebooks": DEMO_ROOT=DEMO_ROOT.parent
TRACK_ROOT=DEMO_ROOT.parent
sys.path.insert(0,str(TRACK_ROOT/"src"))
from multimodal_common import assert_disjoint, image_uri, live_evaluation_enabled, paid_jobs_enabled, project_clients, resolve_or_create_job, sha256, vision_chat, write_jsonl
load_dotenv(DEMO_ROOT/".env")
ROOT=DEMO_ROOT; OUTPUTS=ROOT/"outputs"; OUTPUTS.mkdir(exist_ok=True)
FOUNDRY_RUN_PAID_JOBS=paid_jobs_enabled(); FOUNDRY_RUN_LIVE_EVALUATION=live_evaluation_enabled(); PROJECT_ENDPOINT=os.getenv("FOUNDRY_PROJECT_ENDPOINT","").strip()
print({"demo_root":str(ROOT),"run_paid_jobs":FOUNDRY_RUN_PAID_JOBS,"run_live_evaluation":FOUNDRY_RUN_LIVE_EVALUATION,"project_configured":bool(PROJECT_ENDPOINT)})

## Verify preserved source artifacts

The committed training and validation JSONL are retained byte-for-byte. The sanitized 80-image split membership preserves the bounded run's train/validation/holdout assignment without absolute paths.

In [ ]:
manifest_path=ROOT/"data"/"hash-manifest.csv"
if manifest_path.exists():
    hash_manifest=pd.read_csv(manifest_path)
    for row in hash_manifest.itertuples(index=False):
        if sha256(ROOT/row.preserved_path)!=row.preserved_sha256: raise ValueError(row.preserved_path)
    display(hash_manifest[["dataset","preserved_path","split","rows","byte_match"]])
else:
    print("No committed generated JSONL existed for this source demo; live preparation is authoritative.")

## 3. Fine-grained image classification

Download Stanford Dogs live, use four fixed breed folders, sort filenames, and assign 12 train / 4 validation / 4 holdout images per class. Managed vision safety preprocessing can reject rows; always record submitted and accepted counts. An optional rejection manifest creates an explicit post-filter JSONL and fails if filtering removes a class.

In [ ]:
DOG_URL="https://www.kaggle.com/api/v1/datasets/download/jessicali9530/stanford-dogs-dataset"
DOG_FOLDERS=("n02085620-Chihuahua","n02092339-Weimaraner","n02109525-Saint_Bernard","n02111277-Newfoundland")
def download_dogs():
    root=ROOT/"data"/"downloads"/"stanford-dogs"; images=root/"stanford_dogs_dataset"/"images"/"Images"
    if images.exists(): return images
    root.mkdir(parents=True,exist_ok=True); archive=root/"stanford-dogs-dataset.zip"; urllib.request.urlretrieve(DOG_URL,archive)
    with zipfile.ZipFile(archive) as z:z.extractall(root/"stanford_dogs_dataset")
    archive.unlink(missing_ok=True)
    if not images.exists(): raise FileNotFoundError(images)
    return images
def dog_label(folder): return folder.split("-",1)[1].replace("_"," ").title()
def dog_manifest(images):
    rows=[]
    for folder in DOG_FOLDERS:
        paths=sorted((images/folder).glob("*.jpg"))[:20]
        if len(paths)<20: raise ValueError(f"Expected 20 images in {folder}")
        for i,path in enumerate(paths): rows.append({"image_id":f"{folder}-{i:02d}","label":dog_label(folder),"path":path,"split":"train" if i<12 else "validation" if i<16 else "holdout"})
    frame=pd.DataFrame(rows); assert_disjoint({k:set(v.image_id) for k,v in frame.groupby("split")}); return frame
def dog_prompt(labels): return "Classify the dog as exactly one of: "+", ".join(labels)+". Return only the breed name."
def write_dogs(frame,path,prompt):
    rows=[]
    for row in frame.itertuples(index=False):
        with Image.open(row.path) as im: uri=image_uri(im,384,82)
        rows.append({"messages":[{"role":"system","content":prompt},{"role":"user","content":[{"type":"image_url","image_url":{"url":uri,"detail":"low"}}]},{"role":"assistant","content":row.label}]})
    return write_jsonl(rows,path)
def prepare_dogs(rejections:Path|None=None):
    manifest=dog_manifest(download_dogs()); labels=sorted(manifest.label.unique()); prompt=dog_prompt(labels); rejected=set()
    if rejections and rejections.exists(): rejected=set(pd.read_csv(rejections).image_id)
    accepted=manifest[~manifest.image_id.isin(rejected)].copy(); coverage=accepted[accepted.split.isin(["train","validation"])].groupby(["split","label"]).size().unstack(fill_value=0)
    if (coverage==0).any().any(): raise ValueError("Safety filtering removed an entire class.")
    files={"train":write_dogs(accepted[accepted.split=="train"],OUTPUTS/"image-classification"/"stanford-dogs-train.jsonl",prompt),"validation":write_dogs(accepted[accepted.split=="validation"],OUTPUTS/"image-classification"/"stanford-dogs-validation.jsonl",prompt)}
    manifest.assign(path=manifest.path.map(lambda x:Path(x).name)).to_csv(OUTPUTS/"image-classification"/"split-manifest.csv",index=False)
    print({"submitted":dict(Counter(manifest.split)),"after_rejection_manifest":dict(Counter(accepted.split)),"coverage":coverage.to_dict()}); return files,manifest,prompt
dog_files,dog_data,dog_system=prepare_dogs()

### Train or reuse and inspect safety attrition

The bounded source run submitted 48 training and 16 validation examples; managed preprocessing accepted 30 and 11. Those rates may change. Training below consumes the exact returned paths, not a rebuilt dataset. Reused jobs are still monitored and must succeed.

In [ ]:
dog_job=None
dog_reuse_id=os.getenv("FOUNDRY_IMAGE_FINE_TUNING_JOB_ID","").strip()
if FOUNDRY_RUN_PAID_JOBS or dog_reuse_id:
    if not PROJECT_ENDPOINT: raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT to submit or monitor a fine-tuning job.")
    with project_clients(PROJECT_ENDPOINT) as (_,dog_client):
        dog_job=resolve_or_create_job(client=dog_client,reuse_id=dog_reuse_id,model="gpt-4o-2024-08-06",train=dog_files["train"],validation=dog_files["validation"],suffix="multimodal-dog-breeds",seed=42,training_type="standard",run_paid_jobs=FOUNDRY_RUN_PAID_JOBS)
else: print("Paid training and existing-job monitoring are disabled.")
print({"recorded_submitted_train":48,"recorded_accepted_train":30,"recorded_submitted_validation":16,"recorded_accepted_validation":11})

### Quality, per-class behavior, and robustness

When `FOUNDRY_RUN_LIVE_EVALUATION=true`, run both deployments against identical holdout images. Report overall accuracy, per-class precision/recall/F1, confusion, and image-size/aspect slices. Slices are diagnostics, not causal proof.

In [ ]:
def norm_label(value,labels):
    cleaned=value.strip().strip("`\"' .").casefold()
    for label in labels:
        if cleaned==label.casefold() or label.casefold() in cleaned:return label
    return value.strip()
def evaluate_dogs(manifest):
    base=os.getenv("FOUNDRY_IMAGE_BASE_DEPLOYMENT",""); ft=os.getenv("FOUNDRY_IMAGE_FINE_TUNED_DEPLOYMENT","")
    if not(PROJECT_ENDPOINT and base and ft): raise RuntimeError("Live evaluation requires the project endpoint and both image deployments.")
    labels=sorted(manifest.label.unique()); rows=[]
    with project_clients(PROJECT_ENDPOINT) as (_,client):
        for sample in manifest[manifest.split=="holdout"].itertuples(index=False):
            with Image.open(sample.path) as im: width,height=im.size; uri=image_uri(im,384,82)
            for kind,deployment in (("base",base),("fine_tuned",ft)):
                pred=norm_label(vision_chat(client,deployment,dog_system,"Classify this image.",[uri],20),labels); rows.append({"image_id":sample.image_id,"label":sample.label,"model_kind":kind,"prediction":pred,"correct":pred==sample.label,"pixel_area":width*height,"aspect_ratio":width/height})
    out=pd.DataFrame(rows); out["size_slice"]=pd.qcut(out.pixel_area,2,labels=["smaller","larger"],duplicates="drop"); out["aspect_slice"]=np.where(out.aspect_ratio.between(.8,1.25),"near_square","wide_or_tall"); out.to_csv(OUTPUTS/"image-classification"/"holdout-predictions.csv",index=False)
    for kind,g in out.groupby("model_kind"):
        print(kind,accuracy_score(g.label,g.prediction)); print(classification_report(g.label,g.prediction,labels=labels,zero_division=0)); display(pd.crosstab(g.label,g.prediction,margins=True)); display(g.groupby(["size_slice","aspect_slice"],observed=True).correct.agg(["count","mean"]))
    return out

### Integrated latency methodology

This replaces the former latency-only notebook. Under the same live-evaluation gate, it retains per-request wall time, count, mean, median, min/max, standard deviation, p90/p95/p99, sequential throughput proxy, CSV output, and a boxplot. Warm-ups and randomized interleaving reduce deployment-order drift. Eight requests are a smoke test; decisions need repeated trials.

In [ ]:
def benchmark_image_latency(manifest,repeats=1,warmups=1):
    base=os.getenv("FOUNDRY_IMAGE_BASE_DEPLOYMENT",""); ft=os.getenv("FOUNDRY_IMAGE_FINE_TUNED_DEPLOYMENT","")
    if not(PROJECT_ENDPOINT and base and ft): raise RuntimeError("Live evaluation requires the project endpoint and both image deployments.")
    labels=sorted(manifest.label.unique()); samples=list(manifest[manifest.split=="holdout"].itertuples(index=False)); deployments=(("base",base),("fine_tuned",ft)); rows=[]
    with project_clients(PROJECT_ENDPOINT) as (_,client):
        with Image.open(samples[0].path) as im:warm=image_uri(im,384,82)
        for _,deployment in deployments:
            for _ in range(warmups): vision_chat(client,deployment,dog_system,"Classify this image.",[warm],20)
        schedule=[(kind,deployment,s) for _ in range(repeats) for s in samples for kind,deployment in deployments]; random.Random(42).shuffle(schedule)
        for kind,deployment,s in schedule:
            with Image.open(s.path) as im:uri=image_uri(im,384,82)
            start=time.perf_counter(); pred=vision_chat(client,deployment,dog_system,"Classify this image.",[uri],20); ms=(time.perf_counter()-start)*1000
            rows.append({"model_kind":kind,"image_id":s.image_id,"latency_ms":ms,"prediction":norm_label(pred,labels)})
    per=pd.DataFrame(rows); grouped=per.groupby("model_kind").latency_ms; summary=grouped.agg(count="count",mean_ms="mean",median_ms="median",min_ms="min",max_ms="max",std_ms="std")
    for q in (.9,.95,.99):summary[f"p{int(q*100)}_ms"]=grouped.quantile(q)
    summary["sequential_throughput_proxy_rps"]=1000/summary.mean_ms; per.to_csv(OUTPUTS/"image-classification"/"latency-per-request.csv",index=False); summary.to_csv(OUTPUTS/"image-classification"/"latency-summary.csv")
    ax=per.boxplot(column="latency_ms",by="model_kind",grid=False); ax.set(title="Base vs fine-tuned client latency",xlabel="Model",ylabel="Latency (ms)"); plt.suptitle(""); plt.tight_layout(); plt.savefig(OUTPUTS/"image-classification"/"latency-boxplot.png",dpi=160); plt.show(); return per,summary
if FOUNDRY_RUN_LIVE_EVALUATION:
    dog_evaluation=evaluate_dogs(dog_data)
    dog_latency_requests,dog_latency_summary=benchmark_image_latency(dog_data)
else:
    print("Live quality and latency evaluation disabled; showing checked-in representative evidence without remote inference.")
    recorded_quality=pd.read_csv(ROOT/"assets"/"quality-recorded.csv"); recorded_latency=pd.read_csv(ROOT/"assets"/"latency-recorded.csv")
    display(recorded_quality[recorded_quality.workload=="image_classification"]); display(recorded_latency); display(Image.open(ROOT/"assets"/"accuracy-historical.png")); display(Image.open(ROOT/"assets"/"latency-historical-boxplot.png"))

### Cost worksheet

Supply current Azure prices rather than embedding stale rates. Separate training, hosting, and inference; replace image-token estimates with measured usage when available.

In [ ]:
def estimate_cost(training_examples,epochs,tokens_per_example,training_per_million,hosting_hours,hosting_per_hour,input_tokens,output_tokens,input_per_million,output_per_million):
    training_tokens=training_examples*epochs*tokens_per_example; training=training_tokens/1e6*training_per_million; hosting=hosting_hours*hosting_per_hour; inference=input_tokens/1e6*input_per_million+output_tokens/1e6*output_per_million
    return {"training_tokens":training_tokens,"training":training,"hosting":hosting,"inference":inference,"total":training+hosting+inference}
print(estimate_cost(48,1,585,float(os.getenv("FOUNDRY_TRAINING_PRICE_PER_MILLION","0")),float(os.getenv("FOUNDRY_HOSTING_HOURS","0")),float(os.getenv("FOUNDRY_HOSTING_PRICE_PER_HOUR","0")),0,0,float(os.getenv("FOUNDRY_INPUT_PRICE_PER_MILLION","0")),float(os.getenv("FOUNDRY_OUTPUT_PRICE_PER_MILLION","0"))))

## Conclusion and next step

The checked-in historical 120-class study gained 9.0 accuracy points and reduced mean/tail latency. The bounded four-class rerun stayed flat at 93.75% but reduced mean latency in an eight-request smoke test. The opt-in live path invokes both quality evaluation and the integrated latency benchmark. Separate historical and bounded evidence, expose safety attrition, and require per-class plus repeated latency evaluation. Next, continue to `../03-video-action-recognition` for ordered-frame preprocessing.

> **Variability:** Accuracy, safety acceptance, latency, tokens, and cost vary with sampling, preprocessing, service load, pricing date, and deployment configuration. Actual results may vary by model version, data, configuration, region availability, and service conditions.